<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/06-complex-nested-data/02-maps.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 6.2 — Maps

Finally parsing `sample_logs.txt` properly: `str_to_map` does in one
expression what Module 2 did with hand-written Python.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("6.2-maps")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

raw = spark.read.text(f"{DATA_DIR}/sample_logs.txt")
raw.show(2, truncate=False)

## Parse the log line: fixed fields + a MAP for the variable tail

In [ ]:
logs = raw.select(
    F.split("value", " ").getItem(2).alias("level"),
    F.split("value", " ").getItem(3).alias("component"),
    # everything after the 4th space is key=value pairs:
    F.str_to_map(
        F.concat_ws(" ", F.slice(F.split("value", " "), 5, 100)),
        " ", "=",
    ).alias("attrs"),
)
logs.show(5, truncate=False)
logs.printSchema()   # attrs: map<string,string> — one value type for everything

## Key access — and what a MISSING key does on YOUR Spark

In [ ]:
logs.select(
    col("attrs")["user"].alias("user"),
    F.try_element_at("attrs", F.lit("latency_ms")).alias("latency"),   # null-safe always
    F.map_contains_key("attrs", "amount").alias("has_amount"),
).show(5)

In [ ]:
# element_at on an absent key: ANSI (Spark 4 default) raises; legacy returns null.
# Run it and see which world you're in:
print("ANSI mode:", spark.conf.get("spark.sql.ansi.enabled"))
try:
    logs.select(F.element_at("attrs", F.lit("no_such_key"))).collect()
    print("-> returned nulls (legacy behavior)")
except Exception as e:
    print("-> raised:", str(e).splitlines()[0][:80])
# production stance: try_element_at when absence is legitimate,
# loud element_at for keys that MUST exist (FAILFAST thinking, lesson 5.0)

## The toolkit: keys/values/filter/transform_values

In [ ]:
logs.select(
    F.map_keys("attrs").alias("keys"),
    F.size("attrs").alias("n"),
    F.map_filter("attrs", lambda k, v: k != "user").alias("no_pii"),
).show(4, truncate=False)

## map -> rows: explode, then it's ordinary analytics

In [ ]:
# which attribute keys appear how often, by component?
logs.select("component", F.explode("attrs").alias("key", "value")) \
    .groupBy("component", "key").count() \
    .orderBy("component", F.col("count").desc()).show(12)

## The promotion pattern: map at the edge, contract in the core

In [ ]:
silver = logs.select(
    "level", "component",
    # promote the keys we actually analyze — typed, named, prunable:
    F.try_element_at("attrs", F.lit("user")).alias("user_id"),
    F.try_element_at("attrs", F.lit("status")).alias("status"),
    F.try_element_at("attrs", F.lit("latency_ms")).cast("int").alias("latency_ms"),
    # keep the long tail for whoever needs it later:
    F.map_filter("attrs", lambda k, v: ~k.isin("user", "status", "latency_ms")).alias("extra"),
)
silver.show(6, truncate=False)

# and now ordinary Module 3 analytics on promoted, TYPED columns:
silver.groupBy("component").agg(F.round(F.avg("latency_ms"), 1).alias("avg_latency")).show()

## Exercises

1. Rebuild `attrs` from exploded rows: explode, then `groupBy` + `map_from_entries(collect_list(struct(key, value)))`. Round trip verified?
2. Per user (from the map!): event count and ERROR count — without promoting anything, purely via map access in aggregations.
3. `transform_values`: uppercase every value in `attrs` while leaving keys alone.
4. Pivot the promoted-vs-map boundary: explode `attrs`, pivot on `["user","action","status"]` keys, and compare the result's schema to `silver`'s. Which approach do you prefer for 3 known keys? For 300 unknown ones?
5. `map_concat` two literal maps sharing a key. What happens under the default `mapKeyDedupPolicy`, and after setting it to `LAST_WIN`?

In [ ]:
# your exercise code here
